# Day 56：全栈容器化部署工程（Dockerfile、Docker Compose 与健康检查打包器）

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第五阶段（Stage 5）——**Day 56**！今天我们将结合 DevOps 赵磊的 **Issue #056**，动手实践并验证三大核心工程能力：
> 1. **生产级多阶段 `Dockerfile` 校验**（`validate_dockerfile`）：强制 `python:3.11-slim`、多阶段构建、非 `root` 用户与 `HEALTHCHECK`；
> 2. **Docker Compose 编排结构校验**（`validate_compose_yaml_structure`）：确保 `zhishu-api`、`qdrant`、`redis` 服务拓扑、持久化卷与 `depends_on` 就绪；
> 3. **`.env` 密钥安全防泄漏扫描**（`check_env_secrets_safety`）：拦截硬编码真实密钥与明文口令。

## 1. 环境初始化与导入 `project_iterations/day_56` 模块

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day56_dir = repo_root / "project_iterations" / "day_56"
if str(day56_dir) not in sys.path:
    sys.path.insert(0, str(day56_dir))

from zhishu_agent import (
    audit_deployment_bundle,
    check_env_secrets_safety,
    generate_compose_yaml,
    generate_dockerfile,
    generate_env_example,
    validate_compose_yaml_structure,
    validate_dockerfile,
)

print("已加载 Day 56 模块路径:", day56_dir)

已加载 Day 56 模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_56


## 2. 对 Day 56 生产交付包执行端到端合规性审计

In [2]:
report = audit_deployment_bundle(
    dockerfile_path=day56_dir / "Dockerfile",
    compose_path=day56_dir / "docker-compose.yml",
    env_path=day56_dir / ".env.example",
)

for section in ("dockerfile", "docker_compose", "env_safety"):
    item = report[section]
    print(f"[{section}] passed={item['passed']} | metadata={item['metadata']}")

assert report["overall_passed"] is True
print("全栈容器化部署包审计 100% 通过！")

[dockerfile] passed=True | metadata={'stages': 2, 'final_user': 'zhishu', 'has_healthcheck': True, 'exposed_ports': ['8000']}
[docker_compose] passed=True | metadata={'top_sections': ['networks', 'services', 'volumes'], 'services': ['qdrant', 'redis', 'zhishu-api'], 'volumes': ['qdrant_storage', 'redis_data', 'zhishu_data', 'zhishu_logs']}
[env_safety] passed=True | metadata={'inspected_keys': ['APP_ENV', 'LOG_LEVEL', 'API_PORT', 'OPENAI_API_KEY', 'OPENAI_BASE_URL', 'QDRANT_URL', 'REDIS_PASSWORD', 'REDIS_URL', 'SQLITE_DB_PATH', 'JWT_SECRET_KEY'], 'key_count': 10}
全栈容器化部署包审计 100% 通过！


## 3. 安全红线拦截演练：模拟缺陷 Dockerfile 与密钥泄漏

In [3]:
bad_dockerfile = """
FROM python:3.10
WORKDIR /app
COPY . .
USER root
"""
df_check = validate_dockerfile(bad_dockerfile)
print("缺陷 Dockerfile 拦截结果 passed =", df_check.passed)
for err in df_check.errors:
    print("  - 错误:", err)
assert df_check.passed is False

leaked_env = "OPENAI_API_KEY=sk-live-1234567890abcdef123456\nREDIS_PASSWORD=Admin123456\n"
env_check = check_env_secrets_safety(leaked_env)
print("\n密钥泄漏拦截结果 passed =", env_check.passed)
for err in env_check.errors:
    print("  - 拦截:", err)
assert env_check.passed is False

缺陷 Dockerfile 拦截结果 passed = False
  - 错误: Dockerfile 必须采用多阶段构建（至少两个 FROM 且包含 AS 阶段别名）
  - 错误: Dockerfile 必须基于 python:3.11-slim 精简镜像构建
  - 错误: Dockerfile 最终生效用户为 'root'，必须切换为非 root 用户
  - 错误: Dockerfile 缺少 HEALTHCHECK 健康检查指令

密钥泄漏拦截结果 passed = False
  - 拦截: 第 1 行检测到真实 OpenAI API Key 泄漏风险
  - 拦截: 第 1 行敏感键 OPENAI_API_KEY 疑似硬编码真实值（请替换为 your_..._here 或 ${VAR} 占位符）
  - 拦截: 第 2 行敏感键 REDIS_PASSWORD 疑似硬编码真实值（请替换为 your_..._here 或 ${VAR} 占位符）


## 4. 使用模板生成器产出标准配置并自检

In [4]:
gen_df = generate_dockerfile()
gen_dc = generate_compose_yaml()
gen_env = generate_env_example()

assert validate_dockerfile(gen_df).passed is True
assert validate_compose_yaml_structure(gen_dc).passed is True
assert check_env_secrets_safety(gen_env).passed is True

print("模板生成器自检全部通过！")
print(f"Dockerfile 行数: {len(gen_df.splitlines())}, Compose 行数: {len(gen_dc.splitlines())}, Env 行数: {len(gen_env.splitlines())}")

模板生成器自检全部通过！
Dockerfile 行数: 20, Compose 行数: 55, Env 行数: 9
